# USD/INR Dataset Preprocessing

This notebook cleans the raw Yahoo Finance USD/INR file and creates a model-ready dataset for the dashboard and predictor.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "usd_inr.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "usd_inr_preprocessed.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH


In [ ]:
raw_df = pd.read_csv(RAW_PATH)

print(raw_df.shape)
raw_df.head()


In [ ]:
def clean_usd_inr(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()

    # Yahoo Finance CSV exports can include metadata rows before the actual dates.
    df = df.rename(columns={"Price": "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).copy()

    numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = (
        df.dropna(subset=["Open", "High", "Low", "Close"])
        .drop_duplicates(subset="Date", keep="last")
        .sort_values("Date")
        .reset_index(drop=True)
    )

    df["Volume"] = df["Volume"].fillna(0).astype("int64")
    return df

clean_df = clean_usd_inr(raw_df)

print(clean_df.shape)
clean_df.head()


In [ ]:
def add_time_series_features(df: pd.DataFrame) -> pd.DataFrame:
    featured = df.copy()

    featured["year"] = featured["Date"].dt.year
    featured["month"] = featured["Date"].dt.month
    featured["quarter"] = featured["Date"].dt.quarter
    featured["day_of_week"] = featured["Date"].dt.dayofweek
    featured["is_month_end"] = featured["Date"].dt.is_month_end.astype(int)
    featured["is_quarter_end"] = featured["Date"].dt.is_quarter_end.astype(int)

    featured["daily_return"] = featured["Close"].pct_change()
    featured["log_return"] = np.log(featured["Close"] / featured["Close"].shift(1))
    featured["high_low_spread"] = featured["High"] - featured["Low"]
    featured["open_close_change"] = featured["Close"] - featured["Open"]

    for window in (7, 14, 30):
        featured[f"close_ma_{window}"] = featured["Close"].rolling(window=window, min_periods=1).mean()
        featured[f"close_std_{window}"] = featured["Close"].rolling(window=window, min_periods=2).std()
        featured[f"return_volatility_{window}"] = featured["daily_return"].rolling(window=window, min_periods=2).std()

    for lag in (1, 7, 14, 30):
        featured[f"close_lag_{lag}"] = featured["Close"].shift(lag)
        featured[f"return_lag_{lag}"] = featured["daily_return"].shift(lag)

    featured["target_next_close"] = featured["Close"].shift(-1)
    featured["target_next_return"] = featured["daily_return"].shift(-1)

    return featured

featured_df = add_time_series_features(clean_df)
featured_df.head()


In [ ]:
model_df = featured_df.dropna().reset_index(drop=True)

quality_report = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "clean_rows",
            "model_ready_rows",
            "duplicate_dates",
            "missing_values",
            "start_date",
            "end_date",
        ],
        "value": [
            len(raw_df),
            len(clean_df),
            len(model_df),
            int(clean_df["Date"].duplicated().sum()),
            int(model_df.isna().sum().sum()),
            clean_df["Date"].min().date(),
            clean_df["Date"].max().date(),
        ],
    }
)

quality_report


In [ ]:
model_df.to_csv(PROCESSED_PATH, index=False)

print(f"Saved preprocessed data to: {PROCESSED_PATH}")
print(model_df.shape)
model_df.head()
